In [1]:
from pathlib import Path                                     # بنتعامل بيه مع الملفات والفولدرات
import torch                                                 # عشان نعرف الجهاز فيه كارت شاشة ولا لأ
from sentence_transformers import SentenceTransformer        # المكتبة اللي بتشغّل BGE-M3

if torch.cuda.is_available():                                # لو فيه كارت NVIDIA (جهاز الويندوز)
    device = "cuda"                                          # نشتغل عليه
elif torch.backends.mps.is_available():                      # لو ماك Apple Silicon (جهازك)
    device = "mps"                                           # نشتغل على كارت الماك
else:                                                        # لو مفيش ده ولا ده
    device = "cpu"                                           # نشتغل على الـ CPU
print("device:", device)                                     # عندك المفروض يطبع mps

ROOT = Path.cwd()                                            # نبدأ من فولدر الـ notebook
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:  # طول ما مش لاقيين فولدر المشروع
    ROOT = ROOT.parent                                       # نطلع فولدر لفوق

model = SentenceTransformer("BAAI/bge-m3", cache_folder=str(ROOT / "models"), device=device)  # نحمّل الموديل مرة واحدة من فولدر models

docs = {                                                     # 3 ملفات صغيرة للتجربة
    "breakfast.txt": "Breakfast is served from 7:00 AM to 10:30 AM in the main restaurant.",
    "spa.txt": "The spa is open daily. A 60 minute treatment is 3,500 EGP.",
    "pets.txt": "Pets are not allowed in the hotel.",
}
names = list(docs.keys())                                    # أسماء الملفات
texts = list(docs.values())                                  # الكلام اللي فيهم، بنفس الترتيب

doc_vecs = model.encode(texts, normalize_embeddings=True)    # كل ملف بيتحوّل لـ 1024 رقم
print(doc_vecs.shape)                                        # المفروض (3, 1024): 3 ملفات × 1024 رقم

questions = [                                                # أسئلة الضيوف اللي هنجرّبها
    "الفطار لحد إمتى؟",                                       # مصري، والملف إنجليزي
    "Can I bring my dog?",                                   # dog والملف مكتوب فيه Pets
    "عايز أعمل مساج",                                         # مساج والملف مكتوب فيه spa و treatment
    "What's the weather tomorrow?",                          # سؤال مالوش علاقة بالفندق
]

for q in questions:                                          # نلف على كل سؤال
    q_vec = model.encode([q], normalize_embeddings=True)     # السؤال بيتحوّل لـ 1024 رقم
    scores = model.similarity(q_vec, doc_vecs)[0]            # الـ cosine بين السؤال وكل ملف
    print(q)                                                 # نطبع السؤال
    for i in range(len(names)):                              # نلف على الملفات
        print("   ", round(float(scores[i]), 2), names[i])   # نطبع الـ cosine واسم الملف

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 11:13:59.481000 38895 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


device: mps


Could not reach the Hub ([Errno 8] nodename nor servname provided, or not known). Using cached commit hash for 'BAAI/bge-m3'.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 49727.79it/s]


(3, 1024)
الفطار لحد إمتى؟
    0.61 breakfast.txt
    0.42 spa.txt
    0.42 pets.txt
Can I bring my dog?
    0.43 breakfast.txt
    0.41 spa.txt
    0.66 pets.txt
عايز أعمل مساج
    0.44 breakfast.txt
    0.55 spa.txt
    0.43 pets.txt
What's the weather tomorrow?
    0.43 breakfast.txt
    0.38 spa.txt
    0.43 pets.txt


In [2]:
import time                                                  # عشان نقيس الوقت

model.half()                                                 # نص الذاكرة
model.max_seq_length = 512                                   # أقصى طول 512 token

texts = []                                                   # الكلام اللي في ملفات الفندق
for path in sorted((ROOT / "knowledge_base").glob("*.txt")): # نلف على الملفات
    texts.append(path.read_text(encoding="utf-8"))           # نقرا كل ملف

t = time.time()                                              # نبدأ العدّاد
model.encode(["test"], normalize_embeddings=True)            # جملة واحدة صغيرة
print("1 sentence:", round(time.time() - t, 1), "s")         # خدت كام ثانية

t = time.time()                                              # نبدأ العدّاد تاني
model.encode(texts[:4], normalize_embeddings=True, batch_size=4)  # أول 4 ملفات بس
print("4 files:", round(time.time() - t, 1), "s")            # خدوا كام ثانية

1 sentence: 1.7 s
4 files: 1.5 s


In [3]:
import time                                                  # عشان نقيس البحث بياخد وقت قد إيه

model.half()                                                 # الموديل يستخدم نص الذاكرة
model.max_seq_length = 512                                   # يقرا أول 512 token بس من كل ملف

KB_DIR = ROOT / "knowledge_base"                             # مكان ملفات الفندق

names = []                                                   # أسماء الملفات بالترتيب
texts = []                                                   # الكلام اللي في كل ملف، بنفس الترتيب
for path in sorted(KB_DIR.glob("*.txt")):                    # نلف على كل ملفات الـ txt بالترتيب الأبجدي
    names.append(path.name)                                  # نحفظ اسم الملف
    texts.append(path.read_text(encoding="utf-8"))           # ونحفظ الكلام اللي فيه
print(len(names), "files")                                   # المفروض 44

doc_vecs = model.encode(texts, normalize_embeddings=True, batch_size=4, show_progress_bar=True)  # الـ 44 ملف بيتحوّلوا لأرقام، 4 ملفات كل مرة
print(doc_vecs.shape)                                        # المفروض (44, 1024)

def dense_search(question, top_k=3):                         # دالة البحث بالمعنى
    q_vec = model.encode([question], normalize_embeddings=True)  # السؤال بيتحوّل لـ 1024 رقم
    scores = model.similarity(q_vec, doc_vecs)[0]            # الـ cosine بين السؤال وكل ملف من الـ 44
    results = []                                             # هنحط فيها (الـ score، اسم الملف)
    for i in range(len(names)):                              # نلف على كل الملفات
        results.append((round(float(scores[i]), 2), names[i]))  # نحط الـ score واسم الملف مع بعض
    results.sort(reverse=True)                               # نرتّب من الأعلى للأقل
    return results[:top_k]                                   # نرجّع أعلى top_k ملفات

questions = [                                                # نفس أسئلة الـ BM25 وكمان أسئلة جديدة
    "What time is breakfast?",
    "How much is the spa?",
    "Can I bring my dog?",
    "الفطار لحد إمتى؟",
    "ممكن أدخل الأوضة بدري؟",                                 # مصري: check-in بدري
    "What's the weather tomorrow?",                          # مالوش علاقة بالفندق
]

for q in questions:                                          # نلف على كل سؤال
    print(q)                                                 # نطبع السؤال
    for score, name in dense_search(q):                      # ندوّر ونلف على النتايج
        print("   ", score, name)                            # نطبع الـ score واسم الملف

start = time.time()                                          # نبدأ العدّاد
dense_search("الفطار لحد إمتى؟")                              # بحث واحد
print("time:", round(time.time() - start, 3), "seconds")     # كام ثانية خد (العقد بيقول أقل من 0.5)

44 files


Batches: 100%|██████████| 11/11 [00:08<00:00,  1.33it/s]


(44, 1024)
What time is breakfast?
    0.59 restaurant.txt
    0.58 room_service.txt
    0.58 breakfast.txt
How much is the spa?
    0.67 spa.txt
    0.54 hotel_facilities.txt
    0.52 pricing_and_rates.txt
Can I bring my dog?
    0.62 pet_policy.txt
    0.5 gym.txt
    0.5 accessibility.txt
الفطار لحد إمتى؟
    0.62 restaurant.txt
    0.58 breakfast.txt
    0.57 room_service.txt
ممكن أدخل الأوضة بدري؟
    0.37 identification_policy.txt
    0.35 pet_policy.txt
    0.35 payment_methods.txt
What's the weather tomorrow?
    0.41 noise_policy.txt
    0.4 special_requests.txt
    0.4 restaurant.txt
time: 0.024 seconds
